In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from datasets import Dataset
from transformers import AutoTokenizer
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
import accelerate

# 1. Load Data

In [2]:
# Let's load both dev and test set
dev_data = pd.read_csv("../data/processed/dev_data.csv")
test_data = pd.read_csv("../data/processed/test_data.csv")

# 2. Create train-validation split

In [3]:
train_data, val_data = train_test_split(dev_data, test_size=0.2, random_state=42, stratify=dev_data["sentiment"])

In [4]:
train_data

,original_index,text,sentiment
3287,3370,The value of the order is EUR 4mn .,neutral
1828,2065,"H+_kan Dahlstr+¦m , head of mobility services ...",positive
3967,504,SysOpen Digia had signed an agreement with the...,positive
67,316,`` The transaction strengthens our position .....,positive
164,2818,"Berling Capital , Umo Capital and Veikko Laine...",neutral
...,...,...,...
3184,1583,Russia wants to utilise its huge forest reserv...,neutral
3648,2653,"The price of the 10,000 kroon par value bonds ...",neutral
2883,3729,Teleste expects to start the deliveries at the...,neutral
223,3972,- The Group 's sales during the period were EU...,negative


# 3. Label encoding
Hugging face trainer API expects label not string. So, we are label encoding the target labels

In [5]:
label_mapping = {
    "positive":0,
    "negative":1,
    "neutral":2,

    
}
train_data["label"] = train_data["sentiment"].map(label_mapping)
val_data["label"] = val_data["sentiment"].map(label_mapping)
test_data["label"] = test_data["sentiment"].map(label_mapping)

In [6]:
train_data["label"].value_counts()

label
2    1936
0     926
1     410
Name: count, dtype: int64

# 4. Converting the dataframes to datasets

In [7]:
#train_data -> train_ds
train_ds = Dataset.from_pandas(train_data,preserve_index=False)
# val_data -> val_ds
val_ds = Dataset.from_pandas(val_data,preserve_index=False)
# test_data -> test_ds
test_ds = Dataset.from_pandas(test_data, preserve_index=False)

In [8]:
train_ds[0]

{'original_index': 3370,
 'text': 'The value of the order is EUR 4mn .',
 'sentiment': 'neutral',
 'label': 2}

# 5. Load the FinBERT tokenizer

In [9]:
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")

In [10]:
# Initialize data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
# let's test the tokenizer
text = "The value of the order is EUR 4mn."
tokens = tokenizer(text)
tokens

{'input_ids': [101, 1996, 3643, 1997, 1996, 2344, 2003, 7327, 2099, 1018, 2213, 2078, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [12]:
# let's tokenize all the datasets
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True
    )
# here, the hf datasets might send 1000 samples as a batch
tokenized_train_ds = train_ds.map(tokenize_function, batched=True)
tokenized_val_ds = val_ds.map(tokenize_function, batched=True)
tokenized_test_ds = test_ds.map(tokenize_function, batched=True)
tokenized_test_ds[0]

Map:   0%|          | 0/3272 [00:00<?, ? examples/s]

Map:   0%|          | 0/819 [00:00<?, ? examples/s]

Map:   0%|          | 0/723 [00:00<?, ? examples/s]

{'original_index': 2636,
 'text': 'The mill will have capacity to produce 500,000 tonnes of pulp per year .',
 'sentiment': 'neutral',
 'label': 2,
 'input_ids': [101,
  1996,
  4971,
  2097,
  2031,
  3977,
  2000,
  3965,
  3156,
  1010,
  2199,
  11000,
  1997,
  16016,
  2566,
  2095,
  1012,
  102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

# 6. Cleaning up the datasets
Because, the trainer needs only the `input_ids`, `token_type_ids`, `attention_mask` and `label` and rest of things we can remove from the dataset

In [13]:
remove_columns = ["original_index", "text", "sentiment"]
tokenized_train_ds = tokenized_train_ds.remove_columns(remove_columns)
tokenized_val_ds = tokenized_val_ds.remove_columns(remove_columns)
tokenized_test_ds = tokenized_test_ds.remove_columns(remove_columns)


In [14]:
tokenized_val_ds

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 819
})

# 7. Fine-tuning

## 7.1 Load the Model

In [15]:
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [16]:
# let's inspect the model
model

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,),

In [17]:
model.config

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "dtype": "float32",
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "id2label": {
    "0": "positive",
    "1": "negative",
    "2": "neutral"
  },
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "label2id": {
    "negative": 1,
    "neutral": 2,
    "positive": 0
  },
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.14.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

## 7.2 Eval metrics & Training Config

In [18]:
# define evaluation metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    accuracy = accuracy_score(labels, predictions)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1
    }

In [19]:
# let's specify training arguments
training_args = TrainingArguments(
    output_dir="./finbert-finetuned", # saving dest. for the training outputs
    eval_strategy="epoch", # evaluate on val set after every epoch
    save_strategy="epoch", # save a checkpoint for every epoch
    learning_rate=2e-5, # small  learning rate to adapt to the exisiting knowledge
    per_device_train_batch_size=16, # take 16 training eg -> fwd pass -> Loss -> Backprop -> update weights
    per_device_eval_batch_size=16, # take 16 validation eg -> fwd pass -> get logits -> calc metrics 
    num_train_epochs=3, # fine-tune for 3 rounds over the dataset
    weight_decay=0.01, # regularization
    load_best_model_at_end=True, # whichever epoch got the highest macro_f1_score that is svaed
    metric_for_best_model="macro_f1" # This is the metric to pick the best model
    
)

# 7.3 Creating Trainer
Trainer is the one that connects everything we did until now
 ```text
    Training Arguments
            ↓
Dataset → Trainer ← Model
            ↑
     compute_metrics()
```

In [20]:
trainer = Trainer(
    model=model, # FinBERT + classification head
    args=training_args, # the training configuration above
    train_dataset=tokenized_train_ds, #our training dataset
    eval_dataset=tokenized_val_ds, # our validation dataset
    compute_metrics=compute_metrics, # the function that returns accuracy and macro_f1
    data_collator=data_collator # dynamic padding, this picks the highest tokens in the batch and replace all the lesser ones with the <PAD> tokens
)

In [21]:
# Let's have a look at the trainer args
trainer.args

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval

In [22]:
# time to fine-tune the model to our dataset
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.284169,0.888889,0.879705
2,No log,0.373613,0.887668,0.881726
3,0.195475,0.398389,0.879121,0.873077


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=615, training_loss=0.17254280772635608, metrics={'train_runtime': 75.351, 'train_samples_per_second': 130.27, 'train_steps_per_second': 8.162, 'total_flos': 295357144841136.0, 'train_loss': 0.17254280772635608, 'epoch': 3.0})

In [24]:
# Now let's test this fine-tuned model on our unseen test_data
test_results = trainer.evaluate(tokenized_test_ds)

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.195475,0.379699,3,0.889350,0.883068


# 8. Save the model and tokenizer

In [25]:
trainer.save_model("../models/finbert-finetuned")
model.save_pretrained("../models/finbert-finetuned")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]